# Charts with pandas and matplotlib

Python stage, lesson P21. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** see that a chart is a figure that holds axes, and read it back from its Axes; choose a bar or a line chart from the question; keep bars honest with a y axis that starts at zero; give a chart a title that states its message, labeled axes and a saved PNG.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

def check_chart(label, name, bars=None, line=None, zero=False, title=None, xlabel=False, ylabel=False, no_bars=False, traps=None):
    """Look inside the Axes called `name`: bar heights, line values, the bottom of the y axis, the title and the labels."""
    ax = globals().get(name)
    if ax is None or not hasattr(ax, "get_title"):
        RESULTS[label] = None
        print(f"{label}: give {name} the Axes of your chart, then run this cell again.")
        return False
    problems = []
    try:
        heights = [round(float(p.get_height()), 2) for p in ax.patches if p.get_height() != 0]  # legend swatches are zero-height patches
        values = [round(float(y), 2) for y in ax.lines[0].get_ydata()] if ax.lines else []
        if bars is not None and not same(heights, [round(float(b), 2) for b in bars]):
            problems.append(f"the bars should have the heights {bars!r}, but they are {heights!r}")
        if no_bars and heights:
            problems.append("a line chart has no bars; this one has " + str(len(heights)))
        if line is not None and not same(values, [round(float(v), 2) for v in line]):
            problems.append(f"the line should pass through {line!r}, but it passes through {values!r}")
        if zero and ax.get_ylim()[0] > 0:
            problems.append(f"the y axis starts at {ax.get_ylim()[0]:g}; bars must start at 0")
        if title is True and not ax.get_title().strip():
            problems.append("the chart has no title")
        if isinstance(title, str) and title.lower() not in ax.get_title().lower():
            problems.append(f"the title should name {title!r}; it is {ax.get_title()!r}")
        if xlabel and not ax.get_xlabel().strip():
            problems.append("the x axis has no label")
        if ylabel and not ax.get_ylabel().strip():
            problems.append("the y axis has no label")
    except Exception as error:
        problems.append(f"the Axes could not be read ({type(error).__name__})")
    if not problems:
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. " + "; ".join(problems) + ".")
    return False

# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, MAGENTA, GOLD, GREEN = "#0072b2", "#d6479a", "#ad9100", "#00a07a"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, MAGENTA, GOLD, GREEN]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# One row for each order line with its amount, store, month and category, as built in lessons P13, P15 and P17.
price_of = dict(zip(products["product_id"], products["unit_price_usd"]))
lines = order_items.assign(price=order_items["product_id"].map(price_of))
lines = lines.assign(amount=lines["quantity"] * lines["price"] - lines["discount_usd"])
sales = lines.merge(orders[["order_id", "store_id", "ordered_at"]], on="order_id", validate="many_to_one")
sales = sales.merge(products[["product_id", "category"]], on="product_id", validate="many_to_one")
sales = sales.assign(month=pd.to_datetime(sales["ordered_at"]).dt.month)
# The sales of each store, from January to June.
sales_by_store = sales.groupby("store_id")["amount"].sum()
store_names = dict(zip(stores["store_id"], stores["store_name"]))

## Unit 1: Figure and axes

**Predict first.** what are the types of `fig` and `ax`, and what are the heights of the three bars?

In [ ]:
fig, ax = plt.subplots()
sales_by_store.plot.bar(ax=ax, rot=0)
print(type(fig).__name__, type(ax).__name__)
print([round(float(p.get_height()), 2) for p in ax.patches])

**Predict first.** the chart above has no title. What will the title, the x label and the y label be?

In [ ]:
print(repr(ax.get_title()), repr(ax.get_xlabel()), repr(ax.get_ylabel()))

## Unit 2: Bar or line: the question decides

**Predict first.** how many lines and how many bars will the Axes hold, and what will the six values of the line be?

In [ ]:
monthly = sales.groupby("month")["amount"].sum()
fig, ax = plt.subplots()
monthly.plot.line(ax=ax, marker="o")
print(len(ax.lines), len(ax.patches))
print(ax.lines[0].get_ydata().round(2).tolist())

## Unit 3: Honest bars start at zero

**Predict first.** where does the y axis start when you do not set it?

In [ ]:
named = sales_by_store.rename(index=store_names)
fig, ax = plt.subplots()
named.plot.bar(ax=ax, rot=0)
print(ax.get_ylim()[0])

**Predict first.** the first number is Harbor's sales divided by Old Town's. The second is how tall Harbor's bar looks next to Old Town's on the cut axis. What will the two numbers be?

In [ ]:
fig, ax = plt.subplots()
named.plot.bar(ax=ax, rot=0)
ax.set_ylim(4000, 9500)
low, high = ax.get_ylim()
heights = [float(p.get_height()) for p in ax.patches]
print(round(heights[2] / heights[0], 2), round((heights[2] - low) / (heights[0] - low), 2))

## Unit 4: Say it: title, labels, file

**Predict first.** what will the title say, and will the file exist?

In [ ]:
fig, ax = plt.subplots()
named.plot.bar(ax=ax, rot=0)
ratio = sales_by_store[1] / sales_by_store[3]
ax.set_title(f"Old Town sold {ratio:.1f} times Harbor's sales")
ax.set_xlabel("Store")
ax.set_ylabel("Sales, January to June (USD)")
fig.savefig("sales_by_store.png", dpi=150)
print(ax.get_title())
print(Path("sales_by_store.png").exists())

## Unit 5: Check: read the chart back

**Predict first.** does the chart of unit 4 pass the four questions: a title, an x label, a y label, an axis that starts at 0?

In [ ]:
print(ax.get_title() != "", ax.get_xlabel() != "", ax.get_ylabel() != "", ax.get_ylim()[0] == 0)

## Finish it

### Exercise 1: Finish it

**Task:** Which category sold the most units at Harbor, store 3? Say it in the title of the chart. `ax_1` is the bar chart of the units sold in each category at Harbor. Add a title that names the category with the most units, a label on the x axis and a label on the y axis with the unit. Expected: the chart in `ax_1`, with the title and both labels.

**Expected:** the Axes `ax_1` with a title that names the leading category and both axis labels.

In [ ]:
units = sales[sales["store_id"] == 3].groupby("category")["quantity"].sum()
fig, ax_1 = plt.subplots()
units.plot.bar(ax=ax_1, rot=0)  # given
# add the title, the x label and the y label here, on ax_1

In [ ]:
_ = check_chart("Exercise 1", "ax_1", bars=[214, 508, 145, 174], title="coffee", xlabel=True, ylabel=True)

<details>
<summary>Hint</summary>

Print `units` to find the leading category. The title is a sentence that names it, not the name of the table.

</details>

<details>
<summary>Solution</summary>

```python
units = sales[sales["store_id"] == 3].groupby("category")["quantity"].sum()
fig, ax_1 = plt.subplots()
units.plot.bar(ax=ax_1, rot=0)
ax_1.set_title("Coffee was the best seller at Harbor")
ax_1.set_xlabel("Category")
ax_1.set_ylabel("Units sold, April to June")
```

The chart has four bars, bakery 214, coffee 508, food 145 and tea 174 units, and the title names coffee. A title such as `Units sold by category` describes the table and leaves the finding to the reader, and the check cell asks for the category's name.

</details>

## On your own

### Exercise 2: On your own

**Task:** Is the average order of a cafe worth about the same everywhere? Before you start, say in one sentence what the chart should show: one bar for each store, the average order value, a bar chart that a reader can compare honestly. The average order value of a store is its sales divided by its orders. Draw the chart on an Axes called `ax_2`, with the store names, a title, and a label on each axis. Write the code in the empty cell of the notebook. Expected: three bars and an axis that starts at 0. Write the code in the empty cell below and set `answer_2`.

**Expected:** the Axes `ax_2`: three bars, the y axis starting at 0, a title and both axis labels.

In [ ]:
_ = check_chart("Exercise 2", "ax_2", bars=[12.54, 12.58, 12.98], zero=True, title=True, xlabel=True, ylabel=True)

<details>
<summary>Hint</summary>

`orders.groupby("store_id").size()` counts the orders of each store, and two Series with the same index divide row by row. Which part of unit 3 decides where the axis starts?

</details>

<details>
<summary>Solution</summary>

```python
order_counts = orders.groupby("store_id").size()
average_order = (sales_by_store / order_counts).rename(index=store_names)
fig, ax_2 = plt.subplots()
average_order.plot.bar(ax=ax_2, rot=0)
ax_2.set_title("The average order is worth about the same in every store")
ax_2.set_xlabel("Store")
ax_2.set_ylabel("Average order value (USD)")
```

The bars are 12.54, 12.58 and 12.98 US dollars: Harbor's order is worth a little more. On an axis from 0 the three bars are nearly equal, which is the finding. Setting the axis to 12.4 to 13.1 makes Harbor's bar look about four times as tall as Old Town's, and the check cell flags an axis that does not start at 0.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague draws the monthly sales of Campus, store 2, as a line. The line goes up and down in a strange order, and the story it tells is wrong. Fix the line that builds `campus`, so that the months are in the order of the calendar. Expected: the chart in `ax_3`, the line passing through the six sales of Campus in the order of the months.

**Expected:** the Axes `ax_3`: a line through six points in calendar order, with a title and both axis labels.

In [ ]:
named_months = sales.assign(month_name=pd.to_datetime(sales["ordered_at"]).dt.strftime("%b"))
campus = named_months[named_months["store_id"] == 2].groupby("month_name")["amount"].sum()  # the colleague's line
fig, ax_3 = plt.subplots()
campus.plot.line(ax=ax_3, marker="o")
ax_3.set_title("Campus sales by month")
ax_3.set_xlabel("Month")
ax_3.set_ylabel("Sales (USD)")

In [ ]:
_ = check_chart("Exercise 3", "ax_3", line=[1518.75, 1508.25, 1445.75, 1252.0, 1688.25, 1590.75], no_bars=True, title=True, xlabel=True, ylabel=True)

<details>
<summary>Hint</summary>

Print `campus.index`. In which order does `groupby` list the month names?

</details>

<details>
<summary>Solution</summary>

```python
named_months = sales.assign(month_name=pd.to_datetime(sales["ordered_at"]).dt.strftime("%b"))
campus = named_months[named_months["store_id"] == 2].groupby("month_name")["amount"].sum().reindex(["Jan", "Feb", "Mar", "Apr", "May", "Jun"])
fig, ax_3 = plt.subplots()
campus.plot.line(ax=ax_3, marker="o")
ax_3.set_title("Campus sales by month")
ax_3.set_xlabel("Month")
ax_3.set_ylabel("Sales (USD)")
```

The six points are 1,518.75, 1,508.25, 1,445.75, 1,252.00, 1,688.25 and 1,590.75. `groupby` sorts the keys, and text sorts alphabetically: April, February, January, June, March, May. A line follows the order of its points, so the colleague's line wandered through the year in the wrong order. Reindexing with the month names in calendar order, or grouping on the month number, fixes it; the check cell compares the points.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")